In [1]:
import pandas as pd

df = pd.read_csv('C://Users//HP//Datasets//fifa_world_cup_2026_player_performance.csv')  # adjust to your actual file name/path

print(df.columns.tolist())   # shows all column names
print(df.head())              # shows first 5 rows so I can see the actual structure

['player_id', 'player_name', 'age', 'nationality', 'team', 'jersey_number', 'position', 'height_cm', 'weight_kg', 'preferred_foot', 'club_name', 'market_value_eur', 'match_id', 'match_date', 'stadium', 'city', 'opponent_team', 'tournament_stage', 'match_result', 'goals_team', 'goals_opponent', 'minutes_played', 'goals', 'assists', 'shots', 'shots_on_target', 'expected_goals_xg', 'expected_assists_xa', 'key_passes', 'successful_passes', 'total_passes', 'pass_accuracy', 'dribbles_attempted', 'successful_dribbles', 'crosses', 'successful_crosses', 'tackles', 'interceptions', 'clearances', 'blocks', 'aerial_duels_won', 'aerial_duels_lost', 'recoveries', 'defensive_actions', 'fouls_committed', 'fouls_suffered', 'yellow_cards', 'red_cards', 'offsides', 'saves', 'save_percentage', 'punches', 'clean_sheet', 'goals_conceded', 'penalty_saves', 'distance_covered_km', 'sprint_distance_km', 'top_speed_kmh', 'accelerations', 'decelerations', 'stamina_score', 'player_rating', 'performance_score', 'of

In [1]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# ============================================
# 1. LOAD DATA
# ============================================
df = pd.read_csv('C://Users//HP//Datasets//fifa_world_cup_2026_player_performance.csv')   # adjust filename/path as needed

# ============================================
# 2. DEDUPLICATE TO ONE ROW PER PLAYER
# ============================================
# Why: tournament-level columns (total_goals_tournament, tournament_rating, etc.)
# repeat on every match-row for a player. Summing/averaging raw df would
# multiply each player's totals by however many matches they played.
# We keep only the FIRST row per player, since tournament totals are identical
# across all of that player's rows anyway.
player_df = df.drop_duplicates(subset='player_id', keep='first')[
    ['player_id', 'player_name', 'team', 'position', 'nationality',
     'total_goals_tournament', 'total_assists_tournament',
     'tournament_rating', 'possession_impact', 'pressure_resistance',
     'creativity_score', 'consistency_score', 'clutch_performance_score',
     'player_of_match_awards']
]

# ============================================
# 3. INITIALIZE APP
# ============================================
app = Dash(__name__)

# ============================================
# 4. LAYOUT
# ============================================
app.layout = html.Div([

    html.H1("FIFA World Cup — Team & Player Comparison", style={'textAlign': 'center'}),

    # ---- SECTION A: TEAM COMPARISON ----
    html.H2("Team Comparison", style={'paddingLeft': '20px'}),

    html.Div([
        dcc.Graph(id='team-rating-bar', style={'width': '49%', 'display': 'inline-block'}),
        dcc.Graph(id='team-goals-bar', style={'width': '49%', 'display': 'inline-block'}),
    ]),

    html.Hr(),

    # ---- SECTION B: PLAYER COMPARISON BY TEAM ----
    html.H2("Player Comparison by Team", style={'paddingLeft': '20px'}),

    html.Div([
        html.Label("Select Team"),
        dcc.Dropdown(
            id='team-filter',
            options=[{'label': t, 'value': t} for t in sorted(player_df['team'].unique())],
            value=sorted(player_df['team'].unique())[0],  # default: first team alphabetically
            clearable=False
        )
    ], style={'width': '30%', 'padding': '10px'}),

    dcc.Graph(id='player-rating-bar'),

    dcc.Graph(id='player-goals-assists-bar'),

])


# ============================================
# 5. CALLBACK — TEAM COMPARISON (not filtered, always shows all teams)
# ============================================
@app.callback(
    Output('team-rating-bar', 'figure'),
    Output('team-goals-bar', 'figure'),
    Input('team-filter', 'value'),  # dummy trigger just to run once on load; charts don't actually use it
)
def update_team_comparison(_):

    # ---- Group deduplicated player data by team ----
    team_stats = player_df.groupby('team').agg(
        avg_rating=('tournament_rating', 'mean'),
        total_goals=('total_goals_tournament', 'sum'),
        total_assists=('total_assists_tournament', 'sum')
    ).reset_index()

    # ---- Chart 1: Average tournament rating by team ----
    rating_fig = px.bar(
        team_stats.sort_values('avg_rating', ascending=True),
        x='avg_rating', y='team', orientation='h',
        title='Average Tournament Rating by Team',
        color='avg_rating', color_continuous_scale='Blues'
    )

    # ---- Chart 2: Total goals by team ----
    goals_fig = px.bar(
        team_stats.sort_values('total_goals', ascending=True),
        x='total_goals', y='team', orientation='h',
        title='Total Goals by Team',
        color='total_goals', color_continuous_scale='Greens'
    )

    return rating_fig, goals_fig


# ============================================
# 6. CALLBACK — PLAYER COMPARISON BY TEAM (responds to dropdown)
# ============================================
@app.callback(
    Output('player-rating-bar', 'figure'),
    Output('player-goals-assists-bar', 'figure'),
    Input('team-filter', 'value'),
)
def update_player_comparison(selected_team):

    # ---- Filter to only players on the selected team ----
    team_players = player_df[player_df['team'] == selected_team]

    # ---- Chart 1: Players ranked by tournament rating ----
    rating_fig = px.bar(
        team_players.sort_values('tournament_rating', ascending=True),
        x='tournament_rating', y='player_name', orientation='h',
        title=f'{selected_team} — Players by Tournament Rating',
        color='tournament_rating', color_continuous_scale='Purples'
    )

    # ---- Chart 2: Players' goals vs assists, grouped bars ----
    melted = team_players.melt(
        id_vars='player_name',
        value_vars=['total_goals_tournament', 'total_assists_tournament'],
        var_name='stat', value_name='value'
    )
    goals_assists_fig = px.bar(
        melted, x='player_name', y='value', color='stat',
        barmode='group',
        title=f'{selected_team} — Goals vs Assists by Player'
    )

    return rating_fig, goals_assists_fig


# ============================================
# 7. RUN
# ============================================
if __name__ == '__main__':
    app.run(debug=True)
    